# Claim type and core claim test (atoms)

Two separate judgements per **atom** (`outputs/atomise_v01/opus/atoms.jsonl`):

- `claim_type`: Fact, View, Forecast or Prediction (plus Assumption for supposed antecedents). Prediction and Forecast must be the author's own expectation; bare declaratives are Facts.
- `core`: TRUE only for the author's headline conclusions, FALSE for supporting detail.

Each of Haiku, Sonnet and Opus gets the whole document, then the atoms grouped under their source sentence. Raw responses are cached in `outputs/claim_core_test/`; delete a file to re-run that model.

In [ ]:
import json
import re
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from incolor.argumentation.backends.claude_agent import ClaudeAgentBackend

OUT = ROOT / "outputs" / "claim_core_test"
OUT.mkdir(parents=True, exist_ok=True)
MODELS = ["haiku", "sonnet", "opus"]
TYPES = ["Fact", "View", "Forecast", "Prediction", "Assumption"]

ATOMISE = ROOT / "outputs/atomise_v01/opus"
DOC = re.sub(r"\A---.*?---\s*", "", (ROOT / "samples/BofA_Sept_FOMC_-_hike_or_bond_spike.txt").read_text(encoding="utf-8"), flags=re.S)

sentences = {m.group(1): m.group(2) for ln in (ATOMISE / "input.txt").read_text(encoding="utf-8").splitlines() if (m := re.match(r"\[(P\d+)\]\s+(.*)", ln))}
atoms = [json.loads(ln) for ln in (ATOMISE / "atoms.jsonl").read_text(encoding="utf-8").splitlines() if ln.strip()]
for a in atoms:
    a["pid"] = a["id"].split("-")[0]
print(len(sentences), "sentences,", len(atoms), "atoms,", len(DOC), "chars of document")

120 sentences, 269 atoms, 13792 chars of document


In [7]:
SYSTEM = """You annotate atomic claims extracted from a sell-side FX/rates research note. You get the full note, then the atoms grouped under the sentence they came from (each atom has the extractor's modality tag as a hint only).

For EACH atom give two independent judgements.

1. claim_type - exactly one of:
- Fact: a statement of what is or was the case, with no judgement by the author: published data, market pricing, history, what someone said or did, scheduled events, definitions, and bare declaratives (including bare statements about what a publication will contain, e.g. "The SEP is 50bps total hikes in '26"). Expectations or forecasts attributed to the market or third parties ("widely expected", "priced") are Facts about those parties.
- View: the author's own analytical judgement, interpretation, causal attribution, risk assessment, characterisation or conditional reasoning, including forward-looking judgements that lack a measurable value, horizon or identifiable event.
- Forecast: the author's own expectation of a quantitatively measurable outcome with a value AND a horizon (levels, ranges or changes in yields, prices, rates, macro data, projections).
- Prediction: the author's own expectation that a specific, identifiable future event will or will not happen (a policy decision, dissent, statement, announcement). It must be the author's own thought, marked by author-voice or expectation cues (the author expects/thinks/guesses, likely, should, will, may). A bare declarative with no such cue is a Fact.
- Assumption: a supposed condition (antecedent of if/assuming/unless/"if the author is right"), not asserted by the author.

2. core - TRUE or FALSE. Core claims are the author's headline conclusions: the few main takeaways the whole note exists to deliver and that the rest of the note supports (the key takeaways section is a strong signal, but only its conclusions, not its supporting data). FALSE for supporting evidence, data, premises, background, caveats, secondary details, scenario branches, restatements of lower-level detail, and all Facts and Assumptions unless a Fact is itself the thesis. Be selective: a note of this length has only a handful of core claims, so when unsure return FALSE. Where a conclusion is restated in several places, mark only the atoms that state it in the most general/headline form.

Return ONLY a JSON array, one object per atom, in input order:
{"id": "P005-1", "claim_type": "Prediction", "core": false, "why": "<=8 words"}
No text outside the JSON."""

blocks = []
for pid, text in sentences.items():
    group = [a for a in atoms if a["pid"] == pid]
    if group:
        blocks.append(f"[{pid}] {text}\n" + "\n".join(f"  {a['id']} [{a['modality']}] {a['text']}" for a in group))

USER = f"<document>\n{DOC}\n</document>\n\n<atoms>\n" + "\n".join(blocks) + "\n</atoms>"


def parse(raw: str) -> list[dict]:
    return json.loads(raw[raw.index("[") : raw.rindex("]") + 1])


def run(model: str) -> dict:
    path = OUT / f"{model}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    backend = ClaudeAgentBackend(model=model)
    raw = backend.generate(SYSTEM, USER, {})
    result = {"raw": raw, "rows": parse(raw), "usage": backend.usage}
    path.write_text(json.dumps(result, indent=1, ensure_ascii=False), encoding="utf-8")
    return result


with ThreadPoolExecutor(len(MODELS)) as pool:
    results = dict(zip(MODELS, pool.map(run, MODELS)))

for m, r in results.items():
    cost = sum(u.get("cost_usd") or 0 for u in r["usage"])
    print(f"{m}: {len(r['rows'])} rows, ${cost:.3f}")

haiku: 269 rows, $0.016
sonnet: 269 rows, $0.193
opus: 269 rows, $0.437


In [8]:
df = pd.DataFrame({"text": {a["id"]: a["text"] for a in atoms}})
for m, r in results.items():
    by_id = {row["id"]: row for row in r["rows"]}
    df[f"{m}_type"] = [
        (by_id[i].get("claim_type") or "?").strip().capitalize() if i in by_id else "MISSING" for i in df.index
    ]
    df[f"{m}_core"] = [bool(by_id[i].get("core")) if i in by_id else False for i in df.index]
    df[f"{m}_why"] = [by_id.get(i, {}).get("why", "") for i in df.index]

T = [f"{m}_type" for m in MODELS]
C = [f"{m}_core" for m in MODELS]
df["type_agree"] = df[T].nunique(axis=1) == 1
df["core_votes"] = df[C].sum(axis=1)

print("claim_type counts")
print(pd.DataFrame({m: df[f"{m}_type"].value_counts() for m in MODELS}).fillna(0).astype(int).to_string())
print("\ncore claims per model:", {m: int(df[f"{m}_core"].sum()) for m in MODELS})
print("core votes (0-3 models):", df.core_votes.value_counts().sort_index().to_dict())
print(f"\ntype agreement (all 3): {df.type_agree.mean():.0%}")
for a, b in [("haiku", "sonnet"), ("haiku", "opus"), ("sonnet", "opus")]:
    print(f"  {a} vs {b}: type {(df[f'{a}_type'] == df[f'{b}_type']).mean():.0%}, core {(df[f'{a}_core'] == df[f'{b}_core']).mean():.0%}")

claim_type counts
            haiku  sonnet  opus
Assumption     35      35    35
Fact           72      67    66
Forecast       25      34    23
Prediction     28      34    36
View          109      99   109

core claims per model: {'haiku': 4, 'sonnet': 5, 'opus': 8}
core votes (0-3 models): {0: 260, 1: 4, 2: 2, 3: 3}

type agreement (all 3): 89%
  haiku vs sonnet: type 91%, core 99%
  haiku vs opus: type 95%, core 98%
  sonnet vs opus: type 92%, core 99%


In [9]:
pd.set_option("display.max_colwidth", 100, "display.width", 250, "display.max_rows", 400)
# Atoms any model marked core, with each model's type and core vote
core = df[df.core_votes > 0].copy()
for m in MODELS:
    core[m] = core[f"{m}_type"].str[:4] + core[f"{m}_core"].map({True: "*", False: ""})
core[["text", "core_votes", *MODELS]]

,text,core_votes,haiku,sonnet,opus
P005-1,The Fed hikes 25bp.,3,Pred*,Pred*,Pred*
P006-1,The SEP is 50bps total hikes in '26.,1,Fact,Fact,Fact*
P007-1,The Fed's choice is simple.,1,View*,View,View
P007-2,The Fed's choice is between a hike and a bond spike.,2,View,View*,View*
P008-1,The bond likes orthodoxy.,1,View,View,View*
P009-1,Warsh delivers a hawkish hike.,3,Pred*,Pred*,Pred*
P009-2,Warsh's hawkish hike supports higher front end rates.,1,View,Fore,View*
P009-3,Warsh's hawkish hike supports twist flattening of the UST curve.,2,View,Fore*,View*
P010-1,"The USD outcome is determined by the Fed's ability to ""out-hawk"" market expectations and other G...",3,View*,View*,View*


In [10]:
# Type disagreements
df.loc[~df.type_agree, ["text", *T]]

,text,haiku_type,sonnet_type,opus_type
P009-2,Warsh's hawkish hike supports higher front end rates.,View,Forecast,View
P009-3,Warsh's hawkish hike supports twist flattening of the UST curve.,View,Forecast,View
P020-1,Orthodoxy is to use rate hikes to slow inflation and quell long end rate spike.,Fact,View,Fact
P020-2,Orthodoxy is not to use buybacks to quell long end rate spike.,Fact,View,Fact
P022-1,The SEP shows 50bps of total hikes in '26.,Fact,Prediction,Prediction
P026-2,The UST curve twist flattens.,View,Forecast,View
P044-2,Policymakers' first estimates for 2029 are very similar to the longer-run projections.,View,Prediction,Prediction
P045-2,The 2026 median dot shows 50bp of hikes this year.,Forecast,Forecast,Prediction
P045-3,The 2026 median dot implies another rate increase in Oct or Dec.,Prediction,View,View
P046-1,The 2027 median shows one cut (3.875%).,Forecast,Forecast,Prediction
